In [1]:
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(batchelor))
suppressPackageStartupMessages(library(argparse))
library(BiocParallel)

here::i_am("mapping/run/mnn/mapping_mnn.R")

# Load mapping functions
source(here::here("mapping/run/mnn/mapping_functions_extended.R"))

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))


#####################
## Define settings ##
#####################

# I/O
io$path2atlas <- io$atlas.basedir
io$path2query <- io$basedir

## START TEST ##
args = list()
args$atlas_stages <- c("E7.5","E7.75","E8.0","E8.25","E8.5")
args$query_samples <- opts$samples
args$query_sce <- io$rna.sce
args$atlas_sce <- io$rna.atlas.sce
args$query_metadata <- paste0(io$basedir,"/results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$atlas_metadata <- io$rna.atlas.metadata
args$test <- FALSE
args$npcs <- 50
args$n_neighbours <- 25
args$use_marker_genes <- FALSE
args$cosine_normalisation <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/mapping/test")
## END TEST ##

BPPARAM = MulticoreParam(detectCores()-2)


here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/01_Eomes_RNA/code



In [2]:
################
## Load query ##
################

# Load cell metadata
meta_query <- fread(args$query_metadata) %>% 
  .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$query_samples]
if (isTRUE(args$test)) meta_query <- head(meta_query,n=1000)

# Load SingleCellExperiment
sce_query <- load_SingleCellExperiment(args$query_sce, cells = meta_query$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_query %>% .[cell%in%colnames(sce_query)] %>% setkey(cell) %>% .[colnames(sce_query)]
stopifnot(tmp$cell == colnames(sce_query))
colData(sce_query) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_query),] %>% DataFrame()

################
## Load atlas ##
################

# Load cell metadata
meta_atlas <- fread(args$atlas_metadata) %>%
  .[stage%in%args$atlas_stages] %>%
  .[,sample:=factor(sample)] 

# Filter
if (isTRUE(args$test)) meta_atlas <- head(meta_atlas,n=1000)

# Load SingleCellExperiment
sce_atlas <- load_SingleCellExperiment(args$atlas_sce, normalise = TRUE, cells = meta_atlas$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_atlas %>% .[cell%in%colnames(sce_atlas)] %>% setkey(cell) %>% .[colnames(sce_atlas)]
stopifnot(tmp$cell == colnames(sce_atlas))
colData(sce_atlas) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()

# Sanity cehcks
stopifnot(sum(is.na(rownames(sce_atlas)))==0)
stopifnot(sum(duplicated(rownames(sce_atlas)))==0)

#####################
## Define gene set ##
#####################

# Get gene metadata
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce_atlas)] %>%
  .[!duplicated(symbol)]

rownames(sce_atlas) = gene_metadata[match(rownames(sce_atlas), ens_id), symbol]

# Imprinted genes
imprint = gene_metadata[c(grep('maternally', gene_metadata$description),
                       grep('paternally', gene_metadata$description)), symbol]
#Other imprinted genes: 
#- Nnat (https://www.genecards.org/cgi-bin/carddisp.pl?gene=NNAT)
#- Grb10 (https://www.genecards.org/cgi-bin/carddisp.pl?gene=GRB10)

# Intersect genes
genes.intersect <- intersect(rownames(sce_query), rownames(sce_atlas))

# Filter some genes manually
genes.intersect <- genes.intersect[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",genes.intersect,invert=T)] # filter out non-informative genes
genes.intersect <- genes.intersect[grep("^Hbb|^Hba",genes.intersect,invert=T)] # test removing Haem genes 
genes.intersect <- genes.intersect[!genes.intersect %in% c(imprint, 'Grb10', 'Nnat')] # remove imprinted genes
genes.intersect <- genes.intersect[!genes.intersect %in% c("Xist", "Tsix")] # remove Xist & Tsix
genes.intersect <- genes.intersect[!genes.intersect=="tomato-td"] # remove tomato itself
genes.intersect <- genes.intersect[!genes.intersect %in% gene_metadata[chr=="chrY",symbol]] # no genes on y-chr 

# Subset SingleCellExperiment objects
sce_query  <- sce_query[genes.intersect,]
sce_atlas <- sce_atlas[genes.intersect,]

In [ ]:
decomp <- modelGeneVar(sce[,sample_metadata[tdTom==FALSE, cell]], block=colData(sce[,sample_metadata[tdTom==FALSE, cell]])$sample) # Only detect HVGs from WT samples
decomp <- decomp[decomp$mean > 0.01,]
hvgs <- decomp[order(decomp$FDR),]

In [3]:
# Compare Ricards function with just this
sce_atlas.pb = aggregateAcrossCells(sce_atlas, id=sce_atlas$celltype, BPPARAM = BPPARAM)


In [ ]:
sce_query.pb = aggregateAcrossCells(sce_query, id=sce_query$celltype, BPPARAM = BPPARAM)

In [ ]:
assay(sce_atlas.pb,"logcounts") <- log(1e6*(sweep(assay(sce_atlas.pb),2,colSums(assay(sce_atlas.pb),na.rm=T),"/"))+1)
assay(sce_query.pb,"logcounts") <- log(1e6*(sweep(assay(sce_query.pb),2,colSums(assay(sce_query.pb),na.rm=T),"/"))+1)